# Unify games_matched_final.xlsx

One value per field and game (matched and fuzzy). Nothing is lost: `<field>_raw` keeps the original value, `<field>_source`
where it comes from, and `<field>_all` every value of every place. Rules: `unify.py` and `fields.yaml`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd() if (Path.cwd() / "unify.py").exists() else Path.cwd() / "catalog_project" / "check_games_matched_final"
sys.path[:0] = [str(HERE), str(HERE.parent)]
import completeness as cp  # noqa: E402
import master_lib as ml  # noqa: E402
import unify as un  # noqa: E402

OUT = HERE / "output"
OUT.mkdir(exist_ok=True)

In [ ]:
games, records = cp.load(HERE.parent / "games_matched_final.xlsx")
fields = cp.load_fields()
games["overall_status"].value_counts()

In [ ]:
unified = un.unify(games, records, fields)
unified.shape

## Check: nothing lost

In [ ]:
check = un.check(unified, games, records, fields)
assert check["lost"].sum() == 0, "a value of the file is not in the unified table"
assert check["attr_kept_%"].dropna().eq(100).all(), "a value of Navid was changed"
check

In [ ]:
missing = un.not_normalized(unified, fields)        # doubtful values: their clean value is in the table, to discuss with the team
print(len(missing), "doubtful values to discuss (they are in the clean table):", int(missing["games"].sum()), "games")
missing.groupby("reason")["games"].sum()

In [ ]:
changed = un.changes(unified, fields)              # every value whose writing changed: check them by eye
numbers = un.numbers_kept(unified, fields)
assert numbers["bad"].sum() == 0, "a number changed its value"
variants = unified[unified["rtp_differs_from_name"]][["game_name", "rtp", "rtp_in_name", "rtp_source", "confirmed"]]
print(len(changed), "changes to check |", len(variants), "games whose RTP is not the one in their name")

## Empty fields

The ratio of empty fields of each attribute and of each category (`group` in `fields.yaml`), on matched and fuzzy games.

In [ ]:
ratios = un.empty_ratio(unified, fields)
by_group = un.empty_by_group(ratios)
by_group

## Write

In [ ]:
FIELDS = [n for n, s in fields["fields"].items() if s.get("unify", True)]
clean = unified[["game_uid", "game_name", "provider_name", "own_studio", "overall_status", "confirmed"]
                + [c for n in FIELDS for c in (n, f"{n}_source") + (("max_win_amount", "max_win_currency") if n == "max_win" else ()) + ((f"{n}_from_text",) if n in un.TEXT_FALLBACK else ())]]

unified.to_parquet(OUT / "games_unified.parquet", index=False)
ml.write_excel(OUT / "games_unified.xlsx", {"clean": clean, "full": unified, "check": check, "changes": changed,
                "empty by group": by_group, "empty by field": ratios, "not normalized": missing, "rtp variants": variants},
               freeze={"clean": "C2", "full": "C2"})
print(f"{len(unified)} games x {len(FIELDS)} fields -> {OUT / 'games_unified.xlsx'}")